# 06 — FNO architecture

A spectral layer applies a learned transformation to selected Fourier
coefficients:

\[
v_{k+1}(x)=\sigma\left(Wv_k(x)+\mathcal{F}^{-1}\left(R_\theta\,\mathcal{F}(v_k)\right)(x)\right).
\]

The implementation in `src/oisst_fno/model.py` is intentionally compact so its
mechanics remain inspectable. Every probe here is seeded, so the shapes and
parameter counts printed below are reproducible rather than incidental.

In [ ]:
import torch

from oisst_fno.experiment import collect_environment, set_global_seed
from oisst_fno.metrics import parameter_count
from oisst_fno.model import FNO2d, TruncatedFourierMix2d

# Every probe below is seeded, so the printed numbers are reproducible and can be
# compared against a later run or across machines.
SEED = 42
set_global_seed(SEED)

environment = collect_environment(packages=("torch", "numpy"))
print("torch:", environment.torch_version, "| commit:", environment.git_commit)

layer = TruncatedFourierMix2d(in_channels=4, out_channels=6, modes_y=8, modes_x=8)
x = torch.randn(2, 4, 48, 64)
y = layer(x)
print("spectral layer:", tuple(x.shape), "->", tuple(y.shape))

In [ ]:
LOOKBACK = 14
# 14 SST channels + 1 ocean-mask channel. Coordinates are appended internally.
MODEL_CONFIG = {
    "in_channels": LOOKBACK + 1,
    "out_channels": 1,
    "width": 48,
    "modes_y": 16,
    "modes_x": 16,
    "depth": 4,
    "padding": 8,
}

set_global_seed(SEED)
model = FNO2d(**MODEL_CONFIG)
print(model)
print(f"trainable parameters: {parameter_count(model):,}")

In [ ]:
set_global_seed(SEED)
probe = torch.randn(2, LOOKBACK + 1, 81, 101)
out = model(probe)
print(tuple(probe.shape), "->", tuple(out.shape))
assert out.shape == (2, 1, 81, 101)

# Same seed, same architecture, same weights: a determinism check on CPU.
set_global_seed(SEED)
again = FNO2d(**MODEL_CONFIG)
set_global_seed(SEED)
probe_again = torch.randn(2, LOOKBACK + 1, 81, 101)
assert torch.equal(probe, probe_again)
assert torch.allclose(model(probe), again(probe_again)), "seeded construction must reproduce"
print("deterministic on CPU under a fixed seed")

## Capacity as a function of the spectral knobs

Notebook `10` ablates `width` and `modes`. Their cost is worth seeing before
interpreting any ablation result: a model that improves with more modes may simply
be a larger model, not evidence that high frequencies carry forecast information.

In [ ]:
# Cost of the two knobs that notebook 10 ablates. Parameter count is dominated by the
# retained Fourier coefficients, which grow with modes_y * modes_x * width^2.
print(f"{'width':>6} {'modes':>6} {'parameters':>14}")
for width in (32, 48, 64):
    for modes in (8, 16, 24):
        variant = FNO2d(**{**MODEL_CONFIG, "width": width, "modes_y": modes, "modes_x": modes})
        print(f"{width:>6} {modes:>6} {parameter_count(variant):>14,}")

### What the architecture does *not* guarantee

- Fourier layers do not make the model physically correct.
- Retaining low modes can smooth high-frequency structure.
- Coordinate channels and padding matter because the regional domain is not
  periodic.
- "Resolution invariant" is a theoretical, operator-level concept that must be
  tested empirically for this implementation.
- Seeding makes CPU construction and forward passes reproducible. On GPU it does
  not guarantee bitwise equality — notebook `07` prints the remaining sources of
  variation.